# PPI Metric Regression Analysis
**ASMI Project — Benchmarking AI Metrics Against Experimental Binding Affinity (K_D)**

This notebook systematically evaluates whether computational metrics (AI-derived and physics-based) 
can predict experimental binding affinity (pK_D = -log10(K_D)) using:

1. **Exploratory Data Analysis** — dataset overview, pK_D distributions, metric availability
2. **Simple Regression** — per-metric Pearson & Spearman correlations + scatter plots
3. **Multiple Linear Regression (OLS)** — combined metric models per subset
4. **Non-linear Models** — Random Forest with feature importance
5. **Summary Table** — all metrics ranked by predictive performance

---
> **Setup**: Place `merged_all.csv` in the same directory as this notebook, or update `DATA_PATH` below.

## 0. Setup & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from scipy.stats import pearsonr, spearmanr

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.inspection import permutation_importance

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ── Configuration ────────────────────────────────────────────────────────────
DATA_PATH = 'merged_all.csv'   # <-- update path if needed

# Metrics to evaluate (column names in the dataset)
ALL_METRICS = [
    'plddt',
    'pae_interaction',
    'iptm',
    'ipSAE_min',
    'esm_pll',
    'rosetta_interface_dG',
    'dG_kcal_mol'
]

# Human-readable labels
METRIC_LABELS = {
    'plddt':               'pLDDT (Global)',
    'pae_interaction':     'PAE Interaction',
    'iptm':                'ipTM',
    'ipSAE_min':           'ipSAE_min (AF3)',
    'esm_pll':             'ESM2 PLL',
    'rosetta_interface_dG':'Rosetta Interface ΔG',
    'dG_kcal_mol':         'ΔG (kcal/mol)'
}

# Color palette per dataset
DATASET_COLORS = {
    'PPB-Affinity':    '#2196F3',
    'PPB-Affinity-AF': '#1565C0',
    'Adaptyv_EGFR_R1': '#FF9800',
    'Adaptyv_EGFR_R2': '#E65100',
    'Adaptyv_NIPAH':   '#4CAF50',
    'Overath':         '#9C27B0'
}

plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
})

print('✓ Setup complete')

## 1. Load & Explore Data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\nColumns: {df.columns.tolist()}')
df.head(3)

In [ ]:
# ── Dataset composition ───────────────────────────────────────────────────────
print('=== Records per dataset ===')
print(df['dataset'].value_counts().to_string())

print('\n=== pKD availability per dataset ===')
pkd_avail = df.groupby('dataset')['pKD'].agg(['count', 'notna']).rename(
    columns={'count': 'total', 'notna': 'with_pKD'}
)
pkd_avail['with_pKD'] = df.groupby('dataset')['pKD'].apply(lambda x: x.notna().sum()).values
print(pkd_avail.to_string())

print('\n=== Metric availability (rows with both metric AND pKD) ===')
rows = []
for m in ALL_METRICS:
    total = df[m].notna().sum()
    with_pkd = (df[m].notna() & df['pKD'].notna()).sum()
    rows.append({'metric': METRIC_LABELS[m], 'total_rows': total, 'rows_with_pKD': with_pkd})
avail_df = pd.DataFrame(rows).set_index('metric')
print(avail_df.to_string())

In [ ]:
# ── pKD distribution by dataset ───────────────────────────────────────────────
datasets_with_pkd = df[df['pKD'].notna()]['dataset'].unique()

fig, axes = plt.subplots(1, len(datasets_with_pkd), figsize=(4 * len(datasets_with_pkd), 4), sharey=False)
if len(datasets_with_pkd) == 1:
    axes = [axes]

for ax, ds in zip(axes, datasets_with_pkd):
    sub = df[(df['dataset'] == ds) & df['pKD'].notna()]['pKD']
    ax.hist(sub, bins=30, color=DATASET_COLORS.get(ds, '#888'), edgecolor='white', alpha=0.85)
    ax.set_title(f'{ds}\n(n={len(sub):,})', fontsize=10)
    ax.set_xlabel('pK_D')
    ax.set_ylabel('Count')
    ax.axvline(sub.median(), color='black', linestyle='--', linewidth=1, label=f'Median={sub.median():.2f}')
    ax.legend(fontsize=8)

plt.suptitle('pK_D Distribution by Dataset', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('fig_pkd_distributions.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fig_pkd_distributions.png')

In [ ]:
# ── Data integrity check: dG_kcal_mol is thermodynamically derived from KD ───
# dG = RT * ln(KD) ≈ -1.364 * pKD at 298K, so correlating it with pKD is circular.
# We EXCLUDE dG_kcal_mol from all regression analyses below.
print('⚠  NOTE: dG_kcal_mol in PPB-Affinity-AF is a thermodynamic transform of KD')
print('   (ΔG = RT·ln(KD) ≈ −1.364 × pKD at 298 K).')
print('   It is excluded from regression — using it would be a circular analysis.')
print('   It is retained in the dataset for reference only.
')

REGRESSION_METRICS = [m for m in ALL_METRICS if m != 'dG_kcal_mol']

# ── Define analysis subsets ───────────────────────────────────────────────────
# Subsets with pKD AND at least one metric
SUBSETS = {
    'De-novo EGFR (R1+R2)': df[df['dataset'].isin(['Adaptyv_EGFR_R1','Adaptyv_EGFR_R2']) & df['pKD'].notna()].copy(),
    'De-novo NIPAH':        df[(df['dataset'] == 'Adaptyv_NIPAH') & df['pKD'].notna()].copy(),
    'PPB-Affinity-AF':      df[(df['dataset'] == 'PPB-Affinity-AF') & df['pKD'].notna()].copy(),
    'PPB-Affinity (full)':  df[(df['dataset'] == 'PPB-Affinity') & df['pKD'].notna()].copy(),
    'All with pKD':         df[df['pKD'].notna()].copy()
}

for name, sub in SUBSETS.items():
    metrics_avail = [m for m in ALL_METRICS if sub[m].notna().sum() >= 5]
    print(f"[{name}] n={len(sub):,} | metrics available: {[METRIC_LABELS[m] for m in metrics_avail]}")

## 2. Simple Regression — Per-Metric Correlations with pK_D

For each metric × subset combination with sufficient data (n ≥ 5), we compute:
- **Pearson r** — linear correlation
- **Spearman ρ** — rank-based correlation (robust to outliers)
- **R²** — variance explained by simple linear fit
- **p-value** for both correlation tests

In [ ]:
def simple_regression_stats(x, y, metric_name, subset_name):
    """Compute correlation stats and OLS for one metric vs pKD."""
    mask = x.notna() & y.notna()
    x_, y_ = x[mask].values, y[mask].values
    n = len(x_)
    if n < 5:
        return None

    r, p_r   = pearsonr(x_, y_)
    rho, p_s = spearmanr(x_, y_)

    # OLS
    X_ols = sm.add_constant(x_)
    model = sm.OLS(y_, X_ols).fit()
    r2    = model.rsquared
    slope = model.params[1]
    intercept = model.params[0]

    return {
        'subset': subset_name,
        'metric': METRIC_LABELS.get(metric_name, metric_name),
        'metric_key': metric_name,
        'n': n,
        'pearson_r': round(r, 4),
        'pearson_p': round(p_r, 4),
        'spearman_rho': round(rho, 4),
        'spearman_p': round(p_s, 4),
        'R2': round(r2, 4),
        'slope': round(slope, 4),
        'intercept': round(intercept, 4),
        'x': x_,
        'y': y_
    }

# Run for all subsets × metrics
all_results = []
for subset_name, sub_df in SUBSETS.items():
    for metric in REGRESSION_METRICS:
        res = simple_regression_stats(sub_df[metric], sub_df['pKD'], metric, subset_name)
        if res:
            all_results.append(res)

results_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ['x','y']} for r in all_results])
print(f'Total metric×subset combinations with sufficient data: {len(results_df)}')
results_df.head(10)

In [ ]:
# ── Full correlation table ────────────────────────────────────────────────────
print('=== Simple Regression Results — All Subsets × Metrics ===')

display_cols = ['subset','metric','n','pearson_r','pearson_p','spearman_rho','spearman_p','R2']
print(results_df[display_cols].sort_values(['subset','R2'], ascending=[True, False]).to_string(index=False))

In [ ]:
# ── Scatter plots: one figure per subset ─────────────────────────────────────
def plot_scatter_grid(subset_name, sub_results):
    """Grid of scatter plots for one subset, one panel per metric."""
    if not sub_results:
        return
    ncols = min(3, len(sub_results))
    nrows = int(np.ceil(len(sub_results) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4.5 * nrows))
    axes = np.array(axes).flatten()

    ds_color = '#2196F3'  # default
    # try to pick a color from the subset
    for ds, c in DATASET_COLORS.items():
        if ds.lower() in subset_name.lower():
            ds_color = c
            break

    for i, res in enumerate(sub_results):
        ax = axes[i]
        x_, y_ = res['x'], res['y']
        ax.scatter(x_, y_, alpha=0.55, s=25, color=ds_color, edgecolors='white', linewidths=0.3)

        # Regression line
        x_line = np.linspace(x_.min(), x_.max(), 200)
        y_line = res['slope'] * x_line + res['intercept']
        ax.plot(x_line, y_line, color='#E53935', linewidth=1.8, zorder=5)

        # 95% CI band
        n = len(x_)
        se = np.sqrt(np.sum((y_ - (res['slope']*x_ + res['intercept']))**2) / (n-2))
        x_mean = x_.mean()
        ci = 1.96 * se * np.sqrt(1/n + (x_line - x_mean)**2 / np.sum((x_ - x_mean)**2))
        ax.fill_between(x_line, y_line - ci, y_line + ci, alpha=0.15, color='#E53935')

        sig_r = '*' if res['pearson_p'] < 0.05 else ''
        sig_s = '*' if res['spearman_p'] < 0.05 else ''
        ax.set_title(
            f"{res['metric']}\n"
            f"r={res['pearson_r']:.3f}{sig_r}  ρ={res['spearman_rho']:.3f}{sig_s}  R²={res['R2']:.3f}  n={n}",
            fontsize=9
        )
        ax.set_xlabel(res['metric'], fontsize=9)
        ax.set_ylabel('pK_D', fontsize=9)
        ax.tick_params(labelsize=8)

    # Hide unused axes
    for j in range(len(sub_results), len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(f'Simple Regression — {subset_name}\n(* = p<0.05)', fontsize=12, y=1.01)
    plt.tight_layout()
    fname = f'fig_scatter_{subset_name.replace(" ","_").replace("/","-")}.png'
    plt.savefig(fname, bbox_inches='tight', dpi=150)
    plt.show()
    print(f'Saved: {fname}')

# Plot per subset (skip full PPB-Affinity to avoid overplotting with 12k+ points)
PLOT_SUBSETS = ['De-novo EGFR (R1+R2)', 'De-novo NIPAH', 'PPB-Affinity-AF']
for subset_name in PLOT_SUBSETS:
    sub_res = [r for r in all_results if r['subset'] == subset_name]
    plot_scatter_grid(subset_name, sub_res)

In [ ]:
# ── Correlation heatmap: Spearman ρ across subsets × metrics ─────────────────
pivot_r   = results_df.pivot_table(index='metric', columns='subset', values='pearson_r')
pivot_rho = results_df.pivot_table(index='metric', columns='subset', values='spearman_rho')
pivot_p   = results_df.pivot_table(index='metric', columns='subset', values='spearman_p')

fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(pivot_rho) * 0.8)))

for ax, pivot, title, cmap in zip(
    axes,
    [pivot_r, pivot_rho],
    ['Pearson r', 'Spearman ρ'],
    ['RdBu', 'RdBu']
):
    im = ax.imshow(pivot.values, cmap=cmap, vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=30, ha='right', fontsize=8)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=8)
    ax.set_title(title, fontsize=11)
    plt.colorbar(im, ax=ax, shrink=0.7)

    # Annotate cells
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.values[i, j]
            if not np.isnan(val):
                # add significance marker from p-table
                try:
                    p_val = pivot_p.values[i, j]
                    sig = '*' if (not np.isnan(p_val) and p_val < 0.05) else ''
                except:
                    sig = ''
                ax.text(j, i, f'{val:.2f}{sig}', ha='center', va='center',
                        fontsize=7, color='black' if abs(val) < 0.6 else 'white')

plt.suptitle('Correlation Heatmap — Metrics vs pK_D\n(* = p<0.05 Spearman)', fontsize=12)
plt.tight_layout()
plt.savefig('fig_correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fig_correlation_heatmap.png')

## 3. Multiple Linear Regression (OLS)

For each subset where multiple metrics are available, we fit a combined OLS model. 
We also check for multicollinearity using the Variance Inflation Factor (VIF).

In [ ]:
def run_multiple_ols(subset_name, sub_df, min_n=10):
    """Fit OLS with all available metrics for a given subset."""
    available = [m for m in REGRESSION_METRICS if sub_df[m].notna().sum() >= 5]
    if len(available) < 2:
        print(f'[{subset_name}] Skipped — fewer than 2 metrics available.')
        return None

    # Keep only rows with pKD and at least one metric
    sub = sub_df[['pKD'] + available].dropna(subset=['pKD'])

    # For each available metric, fill NaN with median (to retain rows where only some metrics are missing)
    # Report separately on complete cases
    complete = sub.dropna(subset=available)
    
    print(f'\n{"="*60}')
    print(f'SUBSET: {subset_name}  |  Complete cases: {len(complete)}  |  Metrics: {[METRIC_LABELS[m] for m in available]}')
    print('='*60)

    if len(complete) < min_n:
        print(f'  → Only {len(complete)} complete cases; running on median-imputed data (n={len(sub)}).')
        for m in available:
            sub[m] = sub[m].fillna(sub[m].median())
        analysis_df = sub
    else:
        analysis_df = complete

    X = analysis_df[available]
    y = analysis_df['pKD']

    # VIF
    print('\n--- Variance Inflation Factors (VIF) ---')
    X_vif = sm.add_constant(X)
    vif_data = pd.DataFrame()
    vif_data['metric'] = X.columns
    vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i+1) for i in range(X.shape[1])]
    vif_data['metric'] = [METRIC_LABELS.get(m, m) for m in X.columns]
    print(vif_data.to_string(index=False))
    if (vif_data['VIF'] > 5).any():
        print('  ⚠  VIF > 5 detected — multicollinearity may inflate coefficient estimates.')

    # OLS
    X_ols = sm.add_constant(X)
    model = sm.OLS(y, X_ols).fit()
    print('\n--- OLS Summary ---')
    print(model.summary())

    return model, available, analysis_df


ols_models = {}
for subset_name, sub_df in SUBSETS.items():
    result = run_multiple_ols(subset_name, sub_df)
    if result:
        ols_models[subset_name] = result

In [ ]:
# ── Coefficient plots for each OLS model ─────────────────────────────────────
def plot_coefficients(model, available_metrics, subset_name):
    params = model.params.drop('const', errors='ignore')
    conf   = model.conf_int().drop('const', errors='ignore')
    labels = [METRIC_LABELS.get(m, m) for m in params.index]
    errors = np.array([(params[i] - conf.iloc[i, 0], conf.iloc[i, 1] - params[i])
                       for i in range(len(params))]).T

    colors = ['#E53935' if p < 0.05 else '#90A4AE' for p in model.pvalues.drop('const', errors='ignore')]

    fig, ax = plt.subplots(figsize=(7, max(3, len(params) * 0.7)))
    y_pos = np.arange(len(params))
    ax.barh(y_pos, params.values, xerr=errors, color=colors, edgecolor='white',
            height=0.5, capsize=4, alpha=0.85)
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel('OLS Coefficient (95% CI)')
    ax.set_title(
        f'OLS Coefficients — {subset_name}\n'
        f'R²={model.rsquared:.3f}  Adj.R²={model.rsquared_adj:.3f}  p(F)={model.f_pvalue:.4f}\n'
        f'(red = p<0.05)',
        fontsize=10
    )
    plt.tight_layout()
    fname = f'fig_ols_coefs_{subset_name.replace(" ","_").replace("/","-")}.png'
    plt.savefig(fname, bbox_inches='tight', dpi=150)
    plt.show()
    print(f'Saved: {fname}')

for subset_name, (model, available, _) in ols_models.items():
    plot_coefficients(model, available, subset_name)

## 4. Non-Linear Models — Random Forest & Gradient Boosting

We fit a **Random Forest** and **Gradient Boosting** regressor per subset with cross-validated R².
Feature importance is used to identify which metrics drive non-linear predictions.

In [ ]:
def run_nonlinear_models(subset_name, sub_df, min_n=15, n_cv=5):
    """Fit RF and GBM with CV, return feature importances."""
    available = [m for m in REGRESSION_METRICS if sub_df[m].notna().sum() >= 5]
    if len(available) < 1:
        print(f'[{subset_name}] Skipped — no metrics available.')
        return None

    sub = sub_df[['pKD'] + available].dropna(subset=['pKD']).copy()
    # Median imputation for missing metric values
    for m in available:
        sub[m] = sub[m].fillna(sub[m].median())
    sub = sub.dropna()

    if len(sub) < min_n:
        print(f'[{subset_name}] Skipped — only {len(sub)} complete rows.')
        return None

    X = sub[available].values
    y = sub['pKD'].values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    cv = KFold(n_splits=min(n_cv, len(sub)//3), shuffle=True, random_state=42)

    results = {}
    for name, model in [
        ('Random Forest',       RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
        ('Gradient Boosting',   GradientBoostingRegressor(n_estimators=200, random_state=42))
    ]:
        cv_r2 = cross_val_score(model, X_scaled, y, cv=cv, scoring='r2')
        cv_mse = cross_val_score(model, X_scaled, y, cv=cv, scoring='neg_mean_squared_error')
        model.fit(X_scaled, y)
        y_pred = model.predict(X_scaled)
        train_r2 = r2_score(y, y_pred)

        # Permutation importance on full set
        perm = permutation_importance(model, X_scaled, y, n_repeats=20, random_state=42)

        results[name] = {
            'cv_r2_mean': cv_r2.mean(),
            'cv_r2_std':  cv_r2.std(),
            'train_r2':   train_r2,
            'cv_rmse':    np.sqrt(-cv_mse.mean()),
            'model':      model,
            'perm_imp_mean': perm.importances_mean,
            'perm_imp_std':  perm.importances_std
        }
        print(f'[{subset_name}] {name}: CV R²={cv_r2.mean():.3f}±{cv_r2.std():.3f}  Train R²={train_r2:.3f}  CV RMSE={np.sqrt(-cv_mse.mean()):.3f}')

    return results, available, sub


nonlinear_results = {}
for subset_name, sub_df in SUBSETS.items():
    if subset_name == 'All with pKD':
        continue  # skip mega-set for speed; dominated by PPB-Affinity with few metrics
    print(f'\n--- {subset_name} ---')
    res = run_nonlinear_models(subset_name, sub_df)
    if res:
        nonlinear_results[subset_name] = res

In [ ]:
# ── Feature importance plots ──────────────────────────────────────────────────
for subset_name, (model_dict, available, sub) in nonlinear_results.items():
    fig, axes = plt.subplots(1, 2, figsize=(12, max(3, len(available) * 0.7)))
    metric_labels = [METRIC_LABELS.get(m, m) for m in available]

    for ax, (model_name, res) in zip(axes, model_dict.items()):
        imp    = res['perm_imp_mean']
        imp_sd = res['perm_imp_std']
        order  = np.argsort(imp)
        colors = ['#E53935' if imp[i] > 0 else '#90A4AE' for i in order]

        ax.barh(
            range(len(order)),
            imp[order],
            xerr=imp_sd[order],
            color=colors,
            edgecolor='white',
            height=0.5,
            capsize=3,
            alpha=0.85
        )
        ax.set_yticks(range(len(order)))
        ax.set_yticklabels([metric_labels[i] for i in order], fontsize=9)
        ax.axvline(0, color='black', linewidth=0.7, linestyle='--')
        ax.set_xlabel('Permutation Importance (ΔR²)')
        ax.set_title(
            f'{model_name}\nCV R²={res["cv_r2_mean"]:.3f}±{res["cv_r2_std"]:.3f}',
            fontsize=10
        )

    fig.suptitle(f'Feature Importance — {subset_name}', fontsize=12)
    plt.tight_layout()
    fname = f'fig_importance_{subset_name.replace(" ","_").replace("/","-")}.png'
    plt.savefig(fname, bbox_inches='tight', dpi=150)
    plt.show()
    print(f'Saved: {fname}')

In [ ]:
# ── Predicted vs Actual plots ─────────────────────────────────────────────────
for subset_name, (model_dict, available, sub) in nonlinear_results.items():
    rf_model = model_dict['Random Forest']['model']

    X = sub[available].copy()
    for m in available:
        X[m] = X[m].fillna(X[m].median())
    scaler = StandardScaler()
    X_s = scaler.fit_transform(X.values)
    y_true = sub['pKD'].values
    y_pred = rf_model.predict(X_s)

    fig, ax = plt.subplots(figsize=(5.5, 5))
    ax.scatter(y_true, y_pred, alpha=0.6, s=30, color='#1565C0', edgecolors='white', linewidths=0.3)
    lims = [min(y_true.min(), y_pred.min()) - 0.5, max(y_true.max(), y_pred.max()) + 0.5]
    ax.plot(lims, lims, 'k--', linewidth=1, label='Perfect fit')
    r2 = r2_score(y_true, y_pred)
    r_p, _ = pearsonr(y_true, y_pred)
    ax.set_xlabel('Actual pK_D', fontsize=10)
    ax.set_ylabel('RF Predicted pK_D', fontsize=10)
    ax.set_title(f'Random Forest — {subset_name}\nTrain R²={r2:.3f}  r={r_p:.3f}  n={len(y_true)}', fontsize=10)
    ax.legend(fontsize=8)
    plt.tight_layout()
    fname = f'fig_rf_predicted_{subset_name.replace(" ","_").replace("/","-")}.png'
    plt.savefig(fname, bbox_inches='tight', dpi=150)
    plt.show()
    print(f'Saved: {fname}')

## 5. Summary Table

A consolidated ranking of all metrics across all subsets.

In [ ]:
# ── Build comprehensive summary ───────────────────────────────────────────────
summary = results_df[['subset','metric','n','pearson_r','pearson_p','spearman_rho','spearman_p','R2']].copy()

# Significance flags
summary['sig_pearson']  = summary['pearson_p'].apply(lambda p: '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else '')))
summary['sig_spearman'] = summary['spearman_p'].apply(lambda p: '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else '')))

print('=== COMPREHENSIVE REGRESSION SUMMARY ===')
print('Significance: * p<0.05  ** p<0.01  *** p<0.001')
print()

for subset in summary['subset'].unique():
    sub_sum = summary[summary['subset'] == subset].sort_values('R2', ascending=False)
    print(f"\n{'─'*70}")
    print(f"  {subset}")
    print(f"{'─'*70}")
    print(f"  {'Metric':<25} {'n':>5} {'Pearson r':>10} {'Sig':>4} {'Spearman ρ':>11} {'Sig':>4} {'R²':>7}")
    print(f"  {'-'*25} {'─'*5} {'─'*10} {'─'*4} {'─'*11} {'─'*4} {'─'*7}")
    for _, row in sub_sum.iterrows():
        print(f"  {row['metric']:<25} {int(row['n']):>5} {row['pearson_r']:>10.3f} {row['sig_pearson']:>4} {row['spearman_rho']:>11.3f} {row['sig_spearman']:>4} {row['R2']:>7.4f}")

# Export to CSV
summary.to_csv('regression_summary.csv', index=False)
print(f'\n✓ Saved full summary table: regression_summary.csv')

In [ ]:
# ── Non-linear model summary ──────────────────────────────────────────────────
print('=== NON-LINEAR MODEL SUMMARY (Cross-validated R²) ===')
print(f"{'Subset':<30} {'Model':<22} {'CV R²':>8} {'±':>4} {'Train R²':>10} {'CV RMSE':>9}")
print('─' * 85)

for subset_name, (model_dict, available, _) in nonlinear_results.items():
    for model_name, res in model_dict.items():
        print(f"{subset_name:<30} {model_name:<22} {res['cv_r2_mean']:>8.3f} {res['cv_r2_std']:>4.3f} {res['train_r2']:>10.3f} {res['cv_rmse']:>9.3f}")

In [ ]:
# ── Bar chart: R² comparison — simple vs non-linear per subset ───────────────
summary_rows = []

# Best simple regression R² per subset
for subset, grp in results_df.groupby('subset'):
    best_r2 = grp['R2'].max()
    best_m  = grp.loc[grp['R2'].idxmax(), 'metric']
    summary_rows.append({'subset': subset, 'model': f'Best Simple\n({best_m})', 'R2': best_r2, 'type': 'simple'})

# RF CV R² per subset
for subset_name, (model_dict, _, _) in nonlinear_results.items():
    for mname, res in model_dict.items():
        summary_rows.append({'subset': subset_name, 'model': mname + '\n(CV)', 'R2': max(res['cv_r2_mean'], 0), 'type': 'nonlinear'})

comp_df = pd.DataFrame(summary_rows)

subsets_ordered = [s for s in SUBSETS.keys() if s in comp_df['subset'].unique()]
fig, axes = plt.subplots(1, len(subsets_ordered), figsize=(4 * len(subsets_ordered), 5), sharey=True)
if len(subsets_ordered) == 1:
    axes = [axes]

model_colors = {
    'simple':    '#90CAF9',
    'nonlinear': '#E53935'
}

for ax, subset in zip(axes, subsets_ordered):
    sub = comp_df[comp_df['subset'] == subset]
    colors = [model_colors[t] for t in sub['type']]
    bars = ax.barh(sub['model'], sub['R2'], color=colors, edgecolor='white', height=0.5, alpha=0.85)
    ax.set_xlim(0, max(comp_df['R2'].max() + 0.05, 0.3))
    ax.set_title(subset, fontsize=9)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_xlabel('R²')
    for bar, val in zip(bars, sub['R2']):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=7)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(color='#90CAF9', label='Best Simple (Pearson/OLS)'),
                   Patch(color='#E53935', label='Non-linear (RF/GBM CV)')]
axes[-1].legend(handles=legend_elements, loc='lower right', fontsize=8)

plt.suptitle('R² Comparison: Simple vs Non-linear Models per Subset', fontsize=12)
plt.tight_layout()
plt.savefig('fig_r2_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fig_r2_comparison.png')

## 6. Interpretation Notes

Use this cell to record findings as you review the outputs.

**Key questions to answer from results:**
- Which metric has the highest Pearson r / Spearman ρ with pK_D?
- Does the non-linear RF model meaningfully outperform simple regression (suggesting non-linear relationships)?
- Is the weak regression performance consistent across de-novo (EGFR/NIPAH) and natural (PPB-Affinity) subsets?
- Which metric shows significant VIF inflation in the multiple regression — indicating redundancy?

**Expected finding (from paper draft):** 
Classification AUC is strong (≥0.67) while regression R² remains near zero, supporting the **Gatekeeper Hypothesis** — AI metrics function as structural plausibility filters, not continuous affinity predictors.

In [ ]:
# ── Quick summary printout ────────────────────────────────────────────────────
print('=== QUICK FINDINGS SUMMARY ===')
print()
print('Top metric × subset combinations by Pearson |r|:')
top = results_df.assign(abs_r=results_df['pearson_r'].abs()).sort_values('abs_r', ascending=False).head(10)
for _, row in top.iterrows():
    sig = '***' if row['pearson_p'] < 0.001 else ('**' if row['pearson_p'] < 0.01 else ('*' if row['pearson_p'] < 0.05 else 'n.s.'))
    print(f"  {row['metric']:<25} | {row['subset']:<30} | r={row['pearson_r']:>7.3f} {sig:>5} | R²={row['R2']:.4f} | n={int(row['n'])}")

print()
print('Top metric × subset combinations by Spearman |ρ|:')
top_s = results_df.assign(abs_rho=results_df['spearman_rho'].abs()).sort_values('abs_rho', ascending=False).head(10)
for _, row in top_s.iterrows():
    sig = '***' if row['spearman_p'] < 0.001 else ('**' if row['spearman_p'] < 0.01 else ('*' if row['spearman_p'] < 0.05 else 'n.s.'))
    print(f"  {row['metric']:<25} | {row['subset']:<30} | ρ={row['spearman_rho']:>7.3f} {sig:>5} | n={int(row['n'])}")

---
## 7. Metric-Centric Pooled Regression

Rather than analysing each dataset separately, this section pools **all rows sharing a given metric** across datasets to maximise n per metric. This increases statistical power and allows claims about the metric itself rather than a specific dataset.

**Analytical challenge:** pKD distributions differ significantly between datasets (all pairwise Mann-Whitney p < 0.05), so naive pooling would confound metric effects with dataset-membership effects.

We therefore report three complementary approaches:
- **(A) Naive pooled** — raw correlation across all rows (shown for reference, with caveat)
- **(B) Partial correlation** — residualise both metric and pKD on dataset dummies, then correlate residuals
- **(C) Within-dataset stratified** — per-dataset correlations + n-weighted mean r

The partial r (B) is the most interpretable: it measures the metric–affinity relationship *after removing dataset-level mean differences*.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
from scipy.stats import mannwhitneyu

df['protein_type'] = df['dataset'].map({
    'PPB-Affinity':    'natural',
    'PPB-Affinity-AF': 'natural',
    'Adaptyv_EGFR_R1': 'de-novo',
    'Adaptyv_EGFR_R2': 'de-novo',
    'Adaptyv_NIPAH':   'de-novo',
    'Overath':         'de-novo'
})

# ── Confirm pooling concern: pKD differs across datasets ─────────────────────
print('=== pKD distribution per regression-eligible dataset ===')
for ds in ['Adaptyv_EGFR_R1','Adaptyv_EGFR_R2','Adaptyv_NIPAH','PPB-Affinity-AF']:
    sub = df[(df['dataset']==ds) & df['pKD'].notna()]
    if len(sub)==0: continue
    print(f'  {ds:<22}  n={len(sub):>4}  pKD mean={sub["pKD"].mean():.2f}  std={sub["pKD"].std():.2f}  range=[{sub["pKD"].min():.2f}–{sub["pKD"].max():.2f}]')

print()
print('=== Pairwise Mann-Whitney U tests (pKD between datasets) ===')
ds_list = ['Adaptyv_EGFR_R2','Adaptyv_NIPAH','PPB-Affinity-AF']
for i in range(len(ds_list)):
    for j in range(i+1, len(ds_list)):
        a = df[(df['dataset']==ds_list[i]) & df['pKD'].notna()]['pKD']
        b = df[(df['dataset']==ds_list[j]) & df['pKD'].notna()]['pKD']
        stat, p = mannwhitneyu(a, b)
        sig = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'n.s.'))
        print(f'  {ds_list[i]} vs {ds_list[j]}: p={p:.4f} {sig}')
print()
print('⚠  Significant pKD differences detected — dataset-controlled partial r is required for pooled analysis.')

In [ ]:
# ── Core function: metric-centric analysis ────────────────────────────────────
def metric_centric_analysis(metric, df, min_within_n=5):
    """Pool all rows for a metric+pKD pair; return naive, partial, and stratified correlations."""
    mask = df[metric].notna() & df['pKD'].notna()
    sub  = df[mask].copy()
    if len(sub) < 10:
        return None

    # (A) Naive pooled
    r_n, p_n   = pearsonr(sub[metric], sub['pKD'])
    rho_n, ps_n = spearmanr(sub[metric], sub['pKD'])

    # (B) Partial r — residualise on dataset dummies
    dummies = pd.get_dummies(sub['dataset'], drop_first=True).astype(float)
    def resid(y, X):
        if X.shape[1] == 0: return y.values
        return y.values - LinearRegression().fit(X, y).predict(X)
    m_r   = resid(sub[metric], dummies)
    pkd_r = resid(sub['pKD'],  dummies)
    r_p, p_p     = pearsonr(m_r, pkd_r)
    rho_p, ps_p  = spearmanr(m_r, pkd_r)

    # (C) Within-dataset stratified
    within = []
    for ds in sub['dataset'].unique():
        s = sub[sub['dataset']==ds]
        if len(s) < min_within_n: continue
        r_w, p_w = pearsonr(s[metric], s['pKD'])
        rho_w, _ = spearmanr(s[metric], s['pKD'])
        within.append({'dataset': ds, 'n': len(s), 'r': r_w, 'p': p_w, 'rho': rho_w})
    w_r = sum(d['r']*d['n'] for d in within) / sum(d['n'] for d in within) if within else np.nan

    return {
        'metric': metric, 'label': METRIC_LABELS.get(metric, metric),
        'n_total': len(sub), 'datasets': sub['dataset'].unique().tolist(),
        'r_naive': r_n, 'p_naive': p_n, 'rho_naive': rho_n,
        'r_partial': r_p, 'p_partial': p_p, 'rho_partial': rho_p, 'ps_partial': ps_p,
        'R2_partial': r_p**2,
        'weighted_r': w_r,
        'within': within,
        'x_resid': m_r, 'y_resid': pkd_r
    }

# Run for all metrics
mc_results = {}
for m in REGRESSION_METRICS:
    res = metric_centric_analysis(m, df)
    if res:
        mc_results[m] = res
        sig = lambda p: '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'n.s.'))
        print(f"{res['label']:<22} n={res['n_total']:>4} | "
              f"naive r={res['r_naive']:.3f}({sig(res['p_naive'])}) | "
              f"partial r={res['r_partial']:.3f}({sig(res['p_partial'])}) | "
              f"weighted r={res['weighted_r']:.3f}")
print()
print('✓ Metric-centric analysis complete')

In [ ]:
# ── Figure: partial-r residual scatter plots ──────────────────────────────────
n_metrics = len(mc_results)
ncols = min(3, n_metrics)
nrows = int(np.ceil(n_metrics / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5.5*ncols, 4.5*nrows))
axes = np.array(axes).flatten()

for i, (m, res) in enumerate(mc_results.items()):
    ax = axes[i]
    x_, y_ = res['x_resid'], res['y_resid']

    # colour by dataset
    mask = df[m].notna() & df['pKD'].notna()
    ds_col = df[mask]['dataset'].map(DATASET_COLORS).fillna('#888')

    ax.scatter(x_, y_, c=ds_col.values, alpha=0.55, s=28, edgecolors='white', linewidths=0.3)

    # regression line on residuals
    slope, intercept, *_ = np.polyfit(x_, y_, 1), None
    slope = np.polyfit(x_, y_, 1)[0]
    intercept = np.polyfit(x_, y_, 1)[1]
    x_line = np.linspace(x_.min(), x_.max(), 200)
    ax.plot(x_line, slope*x_line + intercept, color='#E53935', linewidth=1.8, zorder=5)

    sig = lambda p: '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'n.s.'))
    ax.set_title(
        f"{res['label']}\n"
        f"partial r={res['r_partial']:.3f}({sig(res['p_partial'])})  "
        f"ρ={res['rho_partial']:.3f}  n={res['n_total']}",
        fontsize=9
    )
    ax.set_xlabel(f'{res["label"]} residual (dataset-adjusted)', fontsize=8)
    ax.set_ylabel('pK_D residual (dataset-adjusted)', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.axhline(0, color='grey', linewidth=0.5, linestyle=':')
    ax.axvline(0, color='grey', linewidth=0.5, linestyle=':')

# Dataset legend
from matplotlib.lines import Line2D
legend_handles = [Line2D([0],[0], marker='o', color='w', markerfacecolor=c, markersize=7, label=ds)
                  for ds, c in DATASET_COLORS.items() if ds in ['Adaptyv_EGFR_R1','Adaptyv_EGFR_R2','Adaptyv_NIPAH','PPB-Affinity-AF']]
axes[0].legend(handles=legend_handles, fontsize=7, loc='upper left')

for j in range(n_metrics, len(axes)): axes[j].set_visible(False)

fig.suptitle('Metric-Centric Pooled Regression (Dataset-Adjusted Residuals)\n* p<0.05  ** p<0.01  *** p<0.001', fontsize=12)
plt.tight_layout()
plt.savefig('fig_metric_centric_partial.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fig_metric_centric_partial.png')

In [ ]:
# ── Within-dataset stratified forest plot ─────────────────────────────────────
# One panel per metric: within-dataset r values with n-labels
fig, axes = plt.subplots(1, n_metrics, figsize=(3.5*n_metrics, 4), sharey=False)
if n_metrics == 1: axes = [axes]

sig = lambda p: '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else ''))

for ax, (m, res) in zip(axes, mc_results.items()):
    within = res['within']
    if not within:
        ax.set_visible(False)
        continue
    labels = [f"{w['dataset']}\n(n={w['n']})" for w in within]
    rs     = [w['r'] for w in within]
    ps     = [w['p'] for w in within]
    colors = [DATASET_COLORS.get(w['dataset'], '#888') for w in within]

    bars = ax.barh(labels, rs, color=colors, edgecolor='white', height=0.5, alpha=0.85)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.axvline(res['weighted_r'], color='black', linewidth=1.2, linestyle='--',
               label=f'Weighted mean r={res["weighted_r"]:.3f}')
    ax.set_xlim(-0.5, 0.9)
    ax.set_title(f'{res["label"]}\npartial r={res["r_partial"]:.3f}', fontsize=9)
    ax.set_xlabel('Within-dataset Pearson r', fontsize=8)
    ax.legend(fontsize=7)
    for bar, r_val, p_val in zip(bars, rs, ps):
        ax.text(r_val + 0.02, bar.get_y() + bar.get_height()/2,
                f'{r_val:.3f}{sig(p_val)}', va='center', fontsize=7)

fig.suptitle('Within-Dataset Correlations per Metric (Stratified Analysis)', fontsize=12)
plt.tight_layout()
plt.savefig('fig_metric_centric_stratified.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fig_metric_centric_stratified.png')

In [ ]:
# ── Summary table: naive vs partial vs stratified ─────────────────────────────
sig = lambda p: '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'n.s.'))

print('=== METRIC-CENTRIC SUMMARY TABLE ===')
print(f"{'Metric':<22} {'n':>5} {'Naive r':>9} {'Partial r':>10} {'Sig':>5} {'ρ partial':>10} {'R² partial':>11} {'Wtd r':>7}")
print('─'*80)
rows = []
for m, res in mc_results.items():
    s = sig(res['p_partial'])
    print(f"{res['label']:<22} {res['n_total']:>5} {res['r_naive']:>9.3f} {res['r_partial']:>10.3f} {s:>5} {res['rho_partial']:>10.3f} {res['R2_partial']:>11.4f} {res['weighted_r']:>7.3f}")
    rows.append({
        'metric': res['label'], 'n_total': res['n_total'],
        'r_naive': round(res['r_naive'],4), 'r_partial': round(res['r_partial'],4),
        'p_partial': round(res['p_partial'],4), 'sig': s,
        'rho_partial': round(res['rho_partial'],4), 'R2_partial': round(res['R2_partial'],4),
        'weighted_r': round(res['weighted_r'],4),
        'datasets': ', '.join(res['datasets'])
    })

mc_summary_df = pd.DataFrame(rows)
mc_summary_df.to_csv('metric_centric_summary.csv', index=False)
print('\n✓ Saved: metric_centric_summary.csv')

In [ ]:
# ── Comparison: dataset-centric vs metric-centric R² ─────────────────────────
# Side-by-side bar chart showing what changes when you pool by metric

# Dataset-centric best R² per metric (from Section 2 results)
dataset_best = {}
for _, row in results_df.iterrows():
    mk = row['metric_key']
    if mk not in dataset_best or row['R2'] > dataset_best[mk]['R2']:
        dataset_best[mk] = {'R2': row['R2'], 'subset': row['subset'], 'n': row['n']}

labels, r2_ds, r2_mc, ns_ds, ns_mc = [], [], [], [], []
for m in REGRESSION_METRICS:
    if m in mc_results and m in dataset_best:
        labels.append(METRIC_LABELS[m])
        r2_ds.append(dataset_best[m]['R2'])
        r2_mc.append(mc_results[m]['R2_partial'])
        ns_ds.append(int(dataset_best[m]['n']))
        ns_mc.append(mc_results[m]['n_total'])

x = np.arange(len(labels))
width = 0.35
fig, ax = plt.subplots(figsize=(9, 4))
b1 = ax.bar(x - width/2, r2_ds, width, label='Best dataset-specific R²', color='#90CAF9', edgecolor='white', alpha=0.9)
b2 = ax.bar(x + width/2, r2_mc, width, label='Metric-centric partial R²', color='#1565C0', edgecolor='white', alpha=0.9)

for bar, r2, n in zip(b1, r2_ds, ns_ds):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002, f'{r2:.3f}\n(n={n})', ha='center', va='bottom', fontsize=7)
for bar, r2, n in zip(b2, r2_mc, ns_mc):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002, f'{r2:.3f}\n(n={n})', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('R²')
ax.set_title('Dataset-specific vs Metric-centric Partial R²\n(metric-centric uses dataset-adjusted residuals)', fontsize=11)
ax.legend(fontsize=9)
ax.set_ylim(0, max(max(r2_ds), max(r2_mc)) * 1.35)
plt.tight_layout()
plt.savefig('fig_r2_ds_vs_mc.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fig_r2_ds_vs_mc.png')

### Key Findings from Metric-Centric Analysis

| Metric | Pooled n | Partial r | Significant? | Interpretation |
|---|---|---|---|---|
| pLDDT | 232 | 0.048 | n.s. | De-novo datasets pull r toward zero; PPB-AF signal (r=0.377) is isolated to natural complexes |
| PAE Interaction | 60 | 0.280 | * | Marginal; driven largely by small EGFR R1 (n=7, r=0.788) |
| ipTM | 153 | −0.029 | n.s. | Opposite signs across EGFR and NIPAH cancel out |
| ipSAE_min | 100 | 0.248 | * | Single-dataset result (NIPAH only); pooling adds no new data |
| ESM2 PLL | 53 | −0.171 | n.s. | Single-dataset result (EGFR R2 only) |

**Main conclusion:** Pooling by metric does not uncover a hidden signal. The partial correlations remain low (max R²=0.079 for PAE Interaction, which is itself fragile given n=7 in one stratum). The strongest result from Section 2 — pLDDT in PPB-Affinity-AF — *disappears* in the pooled analysis because the de-novo datasets dominate by n and show no relationship. This reinforces that any predictive signal is **context-specific** (natural vs de-novo), not a generalised property of the metric.